## Import Datas

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/ICATH 2026/code/data/preprocessed options datas/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + TechnologyUnderlying

In [9]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [10]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [ ]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price



In [17]:
def compute_bs_price(ticker, option_type):
  data = dataset[ticker]["test"]
  test_data = data[data["optionType"] == option_type]

  test_data['BS_price'] = test_data.apply(
    lambda row: black_scholes(
        S=row['underlyingLastPrice'],
        K=row['strike'],
        T=row['timeToMaturity'],
        r=row['riskFreeRate'],
        sigma=row['impliedVolatility'],
        option_type=row['optionType']
    ),
    axis=1
  )

  rmse = np.sqrt(mean_squared_error(test_data['lastPrice'], test_data['BS_price']))
  print(f"ticker: {ticker}, option type: {option_type}, RMSE: {rmse:.3f}")

## option price using black & scholes model

In [20]:
option_type="put"
for ticker in listTickers:
  compute_bs_price(ticker, option_type)

ticker: F, option type: put, RMSE: 0.564
ticker: GM, option type: put, RMSE: 2.744
ticker: LCID, option type: put, RMSE: 1.870
ticker: RIVN, option type: put, RMSE: 1.316
ticker: TSLA, option type: put, RMSE: 35.707
ticker: BAC, option type: put, RMSE: 1.754
ticker: C, option type: put, RMSE: 3.877
ticker: GS, option type: put, RMSE: 29.810
ticker: JPM, option type: put, RMSE: 10.543
ticker: WFC, option type: put, RMSE: 3.293
ticker: ABBV, option type: put, RMSE: 8.464
ticker: JNJ, option type: put, RMSE: 5.162
ticker: MRK, option type: put, RMSE: 3.871
ticker: PFE, option type: put, RMSE: 1.283
ticker: UNH, option type: put, RMSE: 18.893
ticker: AAPL, option type: put, RMSE: 10.780
ticker: AMZN, option type: put, RMSE: 13.574
ticker: GOOGL, option type: put, RMSE: 16.092
ticker: META, option type: put, RMSE: 34.936
ticker: MSFT, option type: put, RMSE: 20.226


In [23]:
option_type="call"
for ticker in listTickers:
  compute_bs_price(ticker, option_type)

ticker: F, option type: call, RMSE: 0.403
ticker: GM, option type: call, RMSE: 3.191
ticker: LCID, option type: call, RMSE: 1.982
ticker: RIVN, option type: call, RMSE: 1.500
ticker: TSLA, option type: call, RMSE: 40.939
ticker: BAC, option type: call, RMSE: 1.815
ticker: C, option type: call, RMSE: 4.169
ticker: GS, option type: call, RMSE: 36.203
ticker: JPM, option type: call, RMSE: 10.528
ticker: WFC, option type: call, RMSE: 3.483
ticker: ABBV, option type: call, RMSE: 6.892
ticker: JNJ, option type: call, RMSE: 4.888
ticker: MRK, option type: call, RMSE: 2.966
ticker: PFE, option type: call, RMSE: 0.692
ticker: UNH, option type: call, RMSE: 18.807
ticker: AAPL, option type: call, RMSE: 10.507
ticker: AMZN, option type: call, RMSE: 13.374
ticker: GOOGL, option type: call, RMSE: 17.929
ticker: META, option type: call, RMSE: 37.972
ticker: MSFT, option type: call, RMSE: 21.906
